# Embedded-policy trajectory analysis

Reads the per-rank `drlrec*.bin` files written by `drl/pol_IO.f` during a
Python-free (embedded) evaluation and works up the **observation**, **action**
and **reward** signals.

The records are written once per control cycle, at the point where all three
are simultaneously valid: `val_obs` still holds the observation that produced
the action, `pol_hold` holds the action that was applied, and the reward
moving average over the cycle has just completed.

**Conventions in this notebook**

* `obs` is the *raw* wall-parallel/normal fluctuation at the sensing plane, in
  solver units — the network divides it by `u_tau` internally, so what you see
  here is what the flow presented, not what the network consumed.
* `act` is the actuation velocity **before** the zero-net-mass-flux correction
  that `znmf_avg` applies, i.e. the raw network output times `ctrl_max_amp`.
* `rwd` columns are `tau_w`, `|p'v|` and `0.5|v^3|` in `net_gain` mode; in
  `dudy` mode only the first column carries information.

## Setup

In [2]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) or ".")
from read_drlrec import load_run

# ---------------------------------------------------------------- palette
# Categorical slots are assigned in fixed order and never cycled. Only the
# first three are used for point/scatter forms, where every pair has to stay
# separable under colour-vision deficiency.
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d8d7d2"
SEQ  = "Blues"      # magnitude: one hue, light -> dark
DIV  = "RdBu_r"     # signed: two hues about a neutral midpoint

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "figure.facecolor": "white",
    "font.size": 10, "axes.labelsize": 11, "axes.titlesize": 11,
    "axes.edgecolor": INK2, "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.grid": True,
    "legend.frameon": False, "lines.linewidth": 1.8,
})

AttributeError: 'RcParams' object has no attribute '_get'

In [ ]:
# ------------------------------------------------------------- run to read
RUN   = os.path.expanduser("~/.cache/nek_solo_rec")   # <- folder with drlrec*.bin
NU    = 1.0 / 2900.0        # kinematic viscosity, 1/|par viscosity|
DUDY  = 11.75               # uncontrolled reference dU/dy
UTAU  = 0.064               # friction velocity used to scale the observation
ALPHA, BETA, GAMMA = 1.0, 1.0, 1.0    # net-gain reward weights

TAU_REF = NU * DUDY

rec = load_run(RUN)
print(f"files    : {len(rec.files)}")
print(f"records  : {rec.nrec}    agents: {rec.nagent}    "
      f"obs components: {rec.obs.shape[2]}")
print(f"time     : {rec.time[0]:.3f} .. {rec.time[-1]:.3f}  "
      f"(dt={rec.dt}, ndrl={rec.ndrl}, rec_freq={rec.rec_freq})")
print(f"tau_ref  : {TAU_REF:.6e}")

# viscous time unit, for a t+ axis
TSTAR = NU / UTAU**2
tp = (rec.time - rec.time[0]) / TSTAR

## 1. Who the agents are

The identity block is carried inside every record file, so the ranks rejoin
without a separate `NODE_INFO.csv` and without knowing the mesh partition.

In [ ]:
print(rec.agents.head())
print()
print("agents per rank :")
print(rec.agents.groupby("nid").size().to_string())
print()
print("agents per policy:")
print(rec.agents.groupby("ipol").size().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.2))
pol = rec.agents["ipol"].to_numpy()
for ip in np.unique(pol):
    m = pol == ip
    ax.scatter(rec.agents["x"][m], rec.agents["z"][m], s=14,
               color=C[int(ip) % len(C)], label=f"policy {ip}",
               edgecolor="white", linewidth=0.4)
ax.set(xlabel="x", ylabel="z", title="Agent layout on the controlled wall")
if len(np.unique(pol)) > 1:
    ax.legend(loc="upper right")
fig.tight_layout()

## 2. Reward

With `rwd_xavg`/`rwd_zavg` enabled in `DRL`, the reward field is plane-averaged
before it is sampled, so **every agent carries the same reward**. The spread
printed below should be at round-off; if it is not, the averaging flags differ
from what this notebook assumes.

In [ ]:
spread = rec.rwd[:, :, 0].std(axis=1).max() / np.abs(rec.rwd[:, :, 0]).mean()
print(f"max across-agent spread in tau_w, relative: {spread:.2e}")

tau = rec.rwd[:, :, 0].mean(axis=1)
pw  = rec.rwd[:, :, 1].mean(axis=1)
v3  = rec.rwd[:, :, 2].mean(axis=1)

R_tau = 1.0 - tau / TAU_REF          # drag-reduction term
R_pw  = -pw / TAU_REF                # pressure-work penalty
R_v3  = -v3 / TAU_REF                # kinetic-energy penalty
R_tot = ALPHA * R_tau + BETA * R_pw + GAMMA * R_v3

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.6, 5.4), sharex=True)

ax = axes[0]
ax.plot(tp, R_tot, color=C[0])
ax.axhline(0.0, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.set(ylabel="net-gain reward",
       title=f"Total reward   (alpha={ALPHA}, beta={BETA}, gamma={GAMMA})")

ax = axes[1]
for y, c, lab in ((R_tau, C[0], r"$R_\tau = 1-\tau_w/\tau_{ref}$"),
                  (R_pw,  C[1], r"$R_{pw} = -|p'v|/\tau_{ref}$"),
                  (R_v3,  C[2], r"$R_{v^3} = -0.5|v^3|/\tau_{ref}$")):
    ax.plot(tp, y, color=c, label=lab)
ax.axhline(0.0, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.set(xlabel=r"$t^+$", ylabel="component", title="Reward components")
ax.legend(loc="best", ncol=3)

fig.tight_layout()

### Drag reduction

The headline number. Discard the start-up transient before averaging — the
control is switched on abruptly at the restart, so the first few hundred
viscous time units are not representative.

In [ ]:
TP_START = 500.0          # discard t+ < TP_START

sel   = tp >= TP_START
label = f"t+ > {TP_START:.0f}"
if sel.sum() < 10:
    # Fall back to the whole record, and say so -- reporting a transient-free
    # number that silently included the transient is worse than no number.
    print(f"WARNING only {sel.sum()} records past t+={TP_START:.0f}; the run "
          f"only reaches t+={tp[-1]:.0f}. Falling back to the whole record, "
          f"which INCLUDES the start-up transient.")
    sel   = slice(None)
    label = "whole record (transient included)"

DR      = 100.0 * (1.0 - tau[sel].mean() / TAU_REF)
DR_full = 100.0 * (1.0 - tau.mean() / TAU_REF)
print(f"drag reduction, {label:<34s}: {DR:+.3f} %")
print(f"drag reduction, {'whole record':<34s}: {DR_full:+.3f} %")
print(f"mean net-gain reward{'':<19s}: {R_tot[sel].mean():+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.2))
dr_t = 100.0 * (1.0 - tau / TAU_REF)
ax.plot(tp, dr_t, color=C[0], label="instantaneous")
run = np.cumsum(dr_t) / np.arange(1, len(dr_t) + 1)
ax.plot(tp, run, color=C[1], label="running mean")
ax.axhline(0.0, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
if np.any(tp >= TP_START):
    ax.axvline(TP_START, color=INK2, linewidth=0.8, linestyle=(0, (1, 3)))
    ax.annotate(f"{DR:+.2f} %", xy=(tp[-1], DR), xytext=(-6, 6),
                textcoords="offset points", ha="right", color=INK)
ax.set(xlabel=r"$t^+$", ylabel="drag reduction [%]",
       title="Drag reduction relative to the uncontrolled reference")
ax.legend(loc="lower right")
fig.tight_layout()

## 3. Action

`act` is the raw network output scaled by `ctrl_max_amp`, before the
zero-net-mass-flux correction. Saturation at the bounds means `tanh` is
railed — worth knowing, because a policy that spends most of its time
saturated is effectively bang-bang.

In [ ]:
amax = np.abs(rec.act).max()
sat  = np.mean(np.abs(rec.act) > 0.99 * amax) * 100
print(f"action range      : {rec.act.min():+.5f} .. {rec.act.max():+.5f}")
print(f"|action| max      : {amax:.5f}")
print(f"saturated samples : {sat:.2f} %  (within 1% of the bound)")
print(f"spatial mean      : {rec.act.mean():+.3e}  "
      f"(pre-ZNMF, so not expected to vanish)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))

ax = axes[0]
ax.hist(rec.act.reshape(-1), bins=80, color=C[0], edgecolor="white",
        linewidth=0.3)
ax.set(xlabel="action", ylabel="count", title="Action distribution")
ax.grid(axis="x", visible=False)

ax = axes[1]
m, s = rec.act.mean(axis=1), rec.act.std(axis=1)
ax.fill_between(tp, m - s, m + s, color=C[0], alpha=0.22, linewidth=0,
                label=r"$\pm$ 1 std across agents")
ax.plot(tp, m, color=C[0], label="mean across agents")
ax.axhline(0.0, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.set(xlabel=r"$t^+$", ylabel="action", title="Action envelope in time")
ax.legend(loc="best")

fig.tight_layout()

In [ ]:
# Space-time map along the spanwise direction, at the x closest to mid-domain.
xs = rec.agents["x"].to_numpy()
zs = rec.agents["z"].to_numpy()
xtarget = np.unique(xs)[len(np.unique(xs)) // 2]
line = np.where(np.isclose(xs, xtarget))[0]
order = np.argsort(zs[line])
line = line[order]

fig, ax = plt.subplots(figsize=(7.6, 3.4))
v = np.abs(rec.act[:, line]).max()
im = ax.pcolormesh(zs[line], tp, rec.act[:, line], cmap=DIV,
                   vmin=-v, vmax=v, shading="nearest", rasterized=True)
ax.grid(False)
ax.set(xlabel="z", ylabel=r"$t^+$",
       title=f"Action along the span at x = {xtarget:.3f}")
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label("action")
cb.outline.set_visible(False)
fig.tight_layout()

## 4. Observation

Two components at the sensing plane. For the channel these are the streamwise
and wall-normal fluctuations `(u', v')`; for a wing they are the wall-tangential
and wall-normal projections.

In [ ]:
names = [r"$u'$", r"$v'$"] if rec.obs.shape[2] == 2 else \
        [f"obs{k+1}" for k in range(rec.obs.shape[2])]

fig, axes = plt.subplots(1, rec.obs.shape[2] + 1,
                         figsize=(4.0 * (rec.obs.shape[2] + 1), 3.4))
for k in range(rec.obs.shape[2]):
    ax = axes[k]
    ax.hist(rec.obs[:, :, k].reshape(-1), bins=80, color=C[k],
            edgecolor="white", linewidth=0.3)
    ax.set(xlabel=names[k], ylabel="count" if k == 0 else None,
           title=f"{names[k]} distribution")
    ax.grid(axis="x", visible=False)

ax = axes[-1]
h = ax.hist2d(rec.obs[:, :, 0].reshape(-1), rec.obs[:, :, 1].reshape(-1),
              bins=70, cmap=SEQ, rasterized=True)
ax.grid(False)
ax.set(xlabel=names[0], ylabel=names[1], title="Joint distribution")
cb = fig.colorbar(h[3], ax=ax, pad=0.02)
cb.set_label("count")
cb.outline.set_visible(False)
fig.tight_layout()

## 5. The learned control law

Binning the recorded action on the observation plane recovers the policy the
network actually flew, straight from the trajectory. For an 8-neuron actor with
a two-component input this is the whole story: it is the complete input-output
map, restricted to the states the flow visited.

Compare it against opposition control, which would be the plane
`action = -v'` — a diagonal stripe pattern with no `u'` dependence.

In [ ]:
u = rec.obs[:, :, 0].reshape(-1)
v = rec.obs[:, :, 1].reshape(-1)
a = rec.act.reshape(-1)

nb = 60
ur = np.percentile(u, [0.5, 99.5])
vr = np.percentile(v, [0.5, 99.5])
sum_, xe, ye = np.histogram2d(u, v, bins=nb, range=[ur, vr], weights=a)
cnt, _, _ = np.histogram2d(u, v, bins=nb, range=[ur, vr])
mean = np.where(cnt > 0, sum_ / np.maximum(cnt, 1), np.nan)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
v0 = np.nanmax(np.abs(mean))
im = ax.pcolormesh(xe, ye, mean.T, cmap=DIV, vmin=-v0, vmax=v0,
                   shading="flat", rasterized=True)
ax.grid(False)
ax.set(xlabel=names[0], ylabel=names[1],
       title="Mean action over the observation plane")
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label("action")
cb.outline.set_visible(False)
fig.tight_layout()

In [ ]:
# Sensitivity to each input separately, with the other held near its median.
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for k, (name, c) in enumerate(zip(names, C)):
    other = 1 - k
    band = np.abs(rec.obs[:, :, other].reshape(-1)
                  - np.median(rec.obs[:, :, other])) < 0.1 * np.std(
                      rec.obs[:, :, other])
    xk = rec.obs[:, :, k].reshape(-1)[band]
    ak = a[band]
    if len(xk) < 50:
        continue
    edges = np.percentile(xk, np.linspace(1, 99, 30))
    idx = np.digitize(xk, edges)
    xc = np.array([xk[idx == i].mean() for i in range(1, len(edges))])
    ac = np.array([ak[idx == i].mean() for i in range(1, len(edges))])
    ax.plot(xc, ac, color=c, marker="o", markersize=4, label=name)
ax.axhline(0.0, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.set(xlabel="observation component", ylabel="mean action",
       title="Action sensitivity, other component held near its median")
ax.legend(loc="best")
fig.tight_layout()

## 6. Export

`to_dataframe()` flattens to one row per (record, agent) with the agent
metadata joined on — convenient for `groupby`/seaborn, but it is
`nrec x nagent` rows, so build it only when you need it.

In [ ]:
df = rec.to_dataframe()
print(df.shape)
df.head()

In [ ]:
# df.to_parquet("drlrec.parquet")      # compact, keeps dtypes
# df.to_csv("drlrec.csv", index=False) # portable, large
# np.savez_compressed("drlrec.npz", time=rec.time, obs=rec.obs,
#                     act=rec.act, rwd=rec.rwd)